In [2]:
import optimize
import numpy as np
import pandas as pd

conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}
annual_risk_free = 0.02

In [36]:
import importlib
importlib.reload(optimize)

<module 'optimize' from '/Users/linanpluimgmail.com/repos/portfolio_optimizing/optimize.py'>

Current portfolio

In [66]:
# My current portfolio
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE" #iShares MSCI World Small Cap UCITS ETF
           ]
weights = np.array([0.074, 0.926])  
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]

risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

current_gm = optimize.geometric_mean_portfolio(weights, mu, cov)
current_sharpe = optimize.sharpe_ratio(weights, mu, cov, risk_free)

print(f'\nCurrent {interval} GM:', round(current_gm, 6))
print("Approx. annualized GM:", round((1 + current_gm) ** anualization_factor - 1, 6))
print(f'Current {interval} Sharpe:', round(current_sharpe, 6))
print("Approx. annualized Sharpe:", round(current_sharpe * np.sqrt(anualization_factor), 6))

[*********************100%***********************]  2 of 2 completed

min date: 2018-05-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.0971   0.1272
arithmetic_return   0.1084   0.1293
volatility          0.1748   0.1348

Current 1mo GM: 0.009876
Approx. annualized GM: 0.125161
Current 1mo Sharpe: 0.227894
Approx. annualized Sharpe: 0.789448


Optimization

In [ ]:
# Settings for optimization
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE", #iShares MSCI World Small Cap UCITS ETF
            "PPFB.DE", #iShares Physical Gold ETC
            "SXRS.DE", #iShares Diversified Commodity Swap UCITS ETF
           ] 
interval = "1mo"  # "1d", "1wk", or "1mo"
start = "2021-07-01" # or none if you want the whole history
end = None # or none if you want the whole history

In [65]:
# Optimize
returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
    start = start,
    end = end
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1
sharpe = optimize.maximize_sharpe_ratio(mu, cov, risk_free)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")

print("\nPortfolio weights:")
print(weights.round(4))

anualization_factor = conversion_to_annual[interval]
print(f"\nOptimal {interval} Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(anualization_factor), 6))
print(f"Optimal {interval} GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** anualization_factor - 1, 6))


[*********************100%***********************]  4 of 4 completed


min date: 2021-08-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  PPFB.DE  SXRS.DE  VWCE.DE
historical_return   0.0823   0.2018   0.1155   0.1166
arithmetic_return   0.0912   0.1958   0.1221   0.1186
volatility          0.1556   0.1473   0.1596   0.1262

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE         0.0000        0.0
PPFB.DE         0.4751        1.0
SXRS.DE         0.1745        0.0
VWCE.DE         0.3504        0.0

Optimal 1mo Sharpe: 0.41753
Approx. annualized Sharpe: 1.446367
Optimal 1mo GM: 0.015424
Approx. annualized GM: 0.201627
